## Building The CV Toolkit

# Unit 2: Extracting a Reusable Toolkit (`cvkit.py`)

Welcome to Unit 2 of our four-unit course on building a panorama stitcher! In the previous unit, we learned the basics of loading and inspecting image files. Now, as our project grows, we want to keep our main stitching logic clean and easy to read.

To achieve this, we are going to extract our common helper functions into a dedicated, reusable file called **`cvkit.py`**. Building a dedicated computer vision toolkit allows us to easily reuse code across our entire project. In this lesson, we will build a suite of tools to safely load, label, and perfectly stack multiple images side-by-side into a single visual "workbench" panel.

---

## Standardizing Image Inputs

As a quick reminder from our previous unit, **OpenCV** treats images as multi-dimensional **NumPy** arrays. An image has dimensions for its height (rows), its width (columns), and its color channels (like Blue, Green, and Red). We already have a function called `resize_long_edge` provided in our toolkit to handle resizing, so let's focus on safely reading and standardizing our image files.

First, we want a reliable way to load our images. Let's create a function called `read_color` in our `cvkit.py` file:

```python
import cv2

def read_color(path):
    image = cv2.imread(path, cv2.IMREAD_COLOR)

```

Here, we use `cv2.imread` to load the image from the given path and force it into a color format using the `cv2.IMREAD_COLOR` flag.

However, if we provide a bad file path, OpenCV does not automatically throw an error; instead, it simply returns `None`. This can cause confusing errors later in our program. Let's add a safety check to explicitly raise an error if the image fails to load:

```python
import cv2

def read_color(path):
    image = cv2.imread(path, cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError(f"Could not read image: {path}")
    return image

```

By explicitly raising a `ValueError`, we ensure our program fails gracefully and tells us exactly which file is missing.

Next, as we process images in future lessons, some of our operations will output 2-dimensional grayscale images (just height and width) instead of 3-dimensional color arrays. Because we want to stack our images side-by-side in our preview panel, they must all have the same number of dimensions. Let's build an `as_bgr` function to standardize this:

```python
def as_bgr(image):
    if image.ndim == 2:
        return cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    return image.copy()

```

This function checks the number of dimensions in the image array using `.ndim`. If it is 2 (meaning it is a grayscale image), we use OpenCV's `cvtColor` to convert it to a 3-channel BGR image. If it already has 3 channels, we simply return a fresh copy of the image.

---

## Adding Labels To Images

When we are inspecting multiple steps of our panorama pipeline, it is incredibly helpful to have text labels directly on the images. Let's build a `label_image` function to accomplish this.

First, we will ensure the image is in color so that our labels can be drawn consistently:

```python
def label_image(image, text):
    output = as_bgr(image)

```

Next, to make sure our text is readable regardless of what is happening in the picture, we will draw a black rectangle in the top-left corner as a background for our text:

```python
def label_image(image, text):
    output = as_bgr(image)
    label_width = min(output.shape[1], max(220, 12 * len(text)))
    cv2.rectangle(output, (0, 0), (label_width, 34), (0, 0, 0), -1)

```

In this code, we dynamically calculate the `label_width` so that it is wide enough to fit our text, but not wider than the image itself (`output.shape[1]`). Then, we use `cv2.rectangle` to draw a solid black box. The coordinates `(0, 0)` represent the top-left corner, and `(label_width, 34)` represent the bottom-right corner of our rectangle. The `(0, 0, 0)` sets the color to black, and `-1` tells OpenCV to fill the rectangle completely solid.

Finally, we will stamp our text in white over the black background:

```python
def label_image(image, text):
    output = as_bgr(image)
    label_width = min(output.shape[1], max(220, 12 * len(text)))
    cv2.rectangle(output, (0, 0), (label_width, 34), (0, 0, 0), -1)
    cv2.putText(
        output,
        text,
        (10, 23),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (255, 255, 255),
        2,
    )
    return output

```

The `cv2.putText` function places our text starting at the coordinates `(10, 23)` inside our black box. We use a simple font (`cv2.FONT_HERSHEY_SIMPLEX`), scale the text to `0.65`, set the color to pure white `(255, 255, 255)`, and set the line thickness to `2`.

---

## Padding And Stacking The Preview Panel

To view our images side-by-side, we will use a NumPy function called `np.hstack` (horizontal stack). However, `np.hstack` has a strict rule: **all image arrays provided to it must have the exact same height**.

If we have one tall image and one short image, we cannot simply stretch or crop the short image, as that would destroy the data we are trying to inspect! Instead, we will build a `pad_to_height` function that adds a blank black space to the bottom of shorter images.

First, we check if the image even needs padding:

```python
import numpy as np

def pad_to_height(image, height):
    if image.shape[0] == height:
        return image

```

Here, `image.shape[0]` gives us the height in pixels. If it already matches the target height, we do nothing and return the image.

If the image is shorter, we need to create a new, blank black canvas that is the correct size:

```python
import numpy as np

def pad_to_height(image, height):
    if image.shape[0] == height:
        return image
    output = np.zeros((height, image.shape[1], 3), dtype=image.dtype)

```

We use `np.zeros` to create an array filled with black pixels. It takes a tuple of our desired dimensions: the target height, the original width (`image.shape[1]`), and 3 color channels. We also ensure it shares the exact same data type (`dtype`) as the original image.

Finally, we paste our original image directly onto the top of this black canvas:

```python
import numpy as np

def pad_to_height(image, height):
    if image.shape[0] == height:
        return image
    output = np.zeros((height, image.shape[1], 3), dtype=image.dtype)
    output[: image.shape[0], : image.shape[1]] = image
    return output

```

Using Python array slicing, `output[: image.shape[0], : image.shape[1]]` selects a region in the top-left of the canvas that perfectly matches the dimensions of our original image. By assigning `image` to this region, the original picture sits at the top, and the extra height at the bottom remains black.

Now we can combine everything into our `make_panel` function. This function takes a list of items (which are pairs of text names and images) and a `max_size` for resizing:

```python
def make_panel(items, max_size=900):
    if not items:
        raise ValueError("make_panel requires at least one image")
    tiles = []
    for name, image in items:
        labeled = label_image(image, name)
        tiles.append(resize_long_edge(labeled, max_size=max_size))

```

We loop through our items, apply our `label_image` function to give them a title, and then use our provided `resize_long_edge` function to scale them down so they fit nicely on our screen. We store these prepared images in a list called `tiles`.

Lastly, we find the tallest image in our list, pad all the images to match that height, and stack them:

```python
def make_panel(items, max_size=900):
    if not items:
        raise ValueError("make_panel requires at least one image")
    tiles = []
    for name, image in items:
        labeled = label_image(image, name)
        tiles.append(resize_long_edge(labeled, max_size=max_size))
    max_height = max(tile.shape[0] for tile in tiles)
    tiles = [pad_to_height(tile, max_height) for tile in tiles]
    return np.hstack(tiles)

```

We use `max()` to find the largest height among all our tiles. Then, we apply our `pad_to_height` function to every tile. Now that they are all the exact same height, `np.hstack(tiles)` safely stitches them together into one large preview image.

---

## A Cleaner Main Script

Because we extracted all of this logic into `cvkit.py`, our main script (**`solution.py`**) becomes beautifully simple.

Let's set up our main script to accept a file path from the command line and import our new custom tools:

```python
import argparse
import cv2
from cvkit import make_panel, read_color

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("path")
    parser.add_argument("--preview-size", type=int, default=900)
    args = parser.parse_args()

```

Notice that we import `make_panel` and `read_color` directly from our `cvkit` module. Next, we will use them to read our image file. Let's prove our toolkit works by creating a panel that displays two identical images side-by-side:

```python
    image = read_color(args.path)
    
    # We pass a list containing two labeled images to our panel maker
    items_to_display = [("original", image), ("duplicate", image)]
    panel = make_panel(items_to_display, max_size=args.preview_size)

```

By passing a list of two items, our `make_panel` function will label the first one `"original"` and the second one `"duplicate"`, resize them both, and safely stack them horizontally.

Finally, we display the generated panel on the screen:

```python
    cv2.imshow("workbench", panel)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()

```

When this script runs, it opens a window called `workbench`. Inside, you will see your image displayed twice, side-by-side, each with a neat black-and-white label in the top-left corner. If you resize the window or pass images of different sizes in the future, the toolkit handles all the messy dimensional math for you.

---

## Summary And Next Steps

In this lesson, we significantly leveled up our project structure by moving our reusable code into a dedicated module (`cvkit.py`). We standardized our image dimensions to safely handle BGR vs grayscale inputs, and we used NumPy array slicing to pad smaller images with black space, ensuring we never lose data or crash when stacking images side-by-side.

Now it is your turn to build out these tools! In the upcoming practice exercises, you will write `cvkit.py` piece by piece — from handling array standardization to writing the panel builder — until you have a fully functioning visual workbench. Let's head over to the exercises and put this toolkit together!

## Loading Color Images from Disk

## Guaranteeing Three Channels for Every Image

It is time to kick off the toolkit that will power the rest of your panorama stitcher. The plan for this unit is to collect handy image helpers inside a single module called cvkit.py, and the very first tool to move in is read_color.

Open cvkit.py and fill in the body of read_color by following the TODO comments:

    Use cv2.imread to load the file at path in color mode and store it in a variable named image.
    Check whether image is None (this occurs when OpenCV cannot read the file) and raise a ValueError with a message that includes the path.
    Return the loaded image.

The solution.py script is already wired up to call your function, so once you finish, you can run it on any image path to see your toolkit in action. Get this small piece right, and the rest of the workbench will snap together smoothly.

```
import cv2


def read_color(path):
    # TODO: Load the image at `path` in color mode using cv2.imread
    # and store the result in a variable called `image`.

    # TODO: If `image` is None (cv2.imread returns None when it fails),
    # raise a ValueError with a helpful message that includes the path.

    # TODO: Return the loaded image.
    pass


def resize_long_edge(image, max_size=900):
    height, width = image.shape[:2]
    scale = min(1.0, max_size / max(height, width))
    if scale == 1.0:
        return image.copy()
    return cv2.resize(image, (int(width * scale), int(height * scale)))

```

Here is the complete implementation for `read_color` inside `cvkit.py`:

```python
import cv2


def read_color(path):
    # Load the image at `path` in color mode using cv2.imread
    image = cv2.imread(path, cv2.IMREAD_COLOR)

    # If `image` is None, raise a ValueError with a helpful message
    if image is None:
        raise ValueError(f"Could not read image: {path}")

    # Return the loaded image
    return image


def resize_long_edge(image, max_size=900):
    height, width = image.shape[:2]
    scale = min(1.0, max_size / max(height, width))
    if scale == 1.0:
        return image.copy()
    return cv2.resize(image, (int(width * scale), int(height * scale)))

```

## Stamping Labels on Every Image

Nice work setting up read_color earlier — your toolkit is starting to take shape. Now, it is time to add another helper that ensures every image plays nicely with the rest of the workbench.

Sometimes, you will load a grayscale image (2D array), and other times, a color image (3D array with BGR channels). The as_bgr function should smooth out that difference so that the rest of your code can always expect a 3-channel BGR image.

Inside cvkit.py, complete the as_bgr function by following the TODOs:

    Check whether the image is 2-dimensional using image.ndim.
    If it is, convert it to BGR using cv2.cvtColor and the cv2.COLOR_GRAY2BGR flag, and then return the result.
    Otherwise, return a copy of the image so that the original is left untouched.

This little helper will save you from many channel-mismatch headaches later on.

```
import cv2


def read_color(path):
    image = cv2.imread(path, cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError(f"Could not read image: {path}")
    return image


def resize_long_edge(image, max_size=900):
    height, width = image.shape[:2]
    scale = min(1.0, max_size / max(height, width))
    if scale == 1.0:
        return image.copy()
    return cv2.resize(image, (int(width * scale), int(height * scale)))


def as_bgr(image):
    # TODO: Check if the image is 2-dimensional (grayscale) using image.ndim.

    # TODO: If it is grayscale, convert it to a 3-channel BGR image using
    # cv2.cvtColor with the cv2.COLOR_GRAY2BGR flag and return the result.

    # TODO: Otherwise, return a copy of the image so the original stays untouched.
    pass

```

Here is the completed `as_bgr` function in `cvkit.py` to convert grayscale images into 3-channel BGR images while keeping existing BGR images safely copied:

```python
import cv2


def read_color(path):
    image = cv2.imread(path, cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError(f"Could not read image: {path}")
    return image


def resize_long_edge(image, max_size=900):
    height, width = image.shape[:2]
    scale = min(1.0, max_size / max(height, width))
    if scale == 1.0:
        return image.copy()
    return cv2.resize(image, (int(width * scale), int(height * scale)))


def as_bgr(image):
    # Check if the image is 2-dimensional (grayscale) using image.ndim
    if image.ndim == 2:
        # Convert grayscale to 3-channel BGR image
        return cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)

    # Otherwise, return a copy of the original image
    return image.copy()

```

## Padding Images to a Target Height

Now that as_bgr guarantees that every image has three channels, you can safely draw colored overlays on top of any input — whether grayscale or color.

In this exercise, you will add a label_image helper to cvkit.py that stamps a small black banner with white text in the top-left corner of an image. This is the function you will use later to tag each panel in your panorama workbench (such as "original", "resized", or "stitched").

Follow the TODO comments inside label_image step by step:

    Normalize the image to BGR using as_bgr and store it as output.
    Compute label_width so that the banner never overflows the image.
    Draw a filled black rectangle and then overlay white text with cv2.putText.
    Return the labeled output.

Once this helper is in place, your toolkit will be ready to annotate every image you display on the workbench.

```
import cv2


def read_color(path):
    image = cv2.imread(path, cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError(f"Could not read image: {path}")
    return image


def resize_long_edge(image, max_size=900):
    height, width = image.shape[:2]
    scale = min(1.0, max_size / max(height, width))
    if scale == 1.0:
        return image.copy()
    return cv2.resize(image, (int(width * scale), int(height * scale)))


def as_bgr(image):
    if image.ndim == 2:
        return cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    return image.copy()


def label_image(image, text):
    # TODO: Normalize the input image to BGR by calling `as_bgr(image)`
    # and store the result in a variable called `output`.

    # TODO: Compute `label_width` as min(output.shape[1], max(220, 12 * len(text)))
    # so the label band never goes wider than the image itself.

    # TODO: Draw a filled black rectangle in the top-left corner using
    # cv2.rectangle with corners (0, 0) and (label_width, 34),
    # color (0, 0, 0), and thickness -1 (to fill it).

    # TODO: Draw the label text on top of the rectangle using cv2.putText
    # at position (10, 23) with cv2.FONT_HERSHEY_SIMPLEX, scale 0.65,
    # color (255, 255, 255), and thickness 2.

    # TODO: Return the labeled `output` image.
    pass

```

Here is the completed `label_image` function added to `cvkit.py`:

```python
import cv2


def read_color(path):
    image = cv2.imread(path, cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError(f"Could not read image: {path}")
    return image


def resize_long_edge(image, max_size=900):
    height, width = image.shape[:2]
    scale = min(1.0, max_size / max(height, width))
    if scale == 1.0:
        return image.copy()
    return cv2.resize(image, (int(width * scale), int(height * scale)))


def as_bgr(image):
    if image.ndim == 2:
        return cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    return image.copy()


def label_image(image, text):
    # Normalize the input image to BGR
    output = as_bgr(image)

    # Compute label_width so the band never exceeds the image width
    label_width = min(output.shape[1], max(220, 12 * len(text)))

    # Draw a filled black rectangle in the top-left corner
    cv2.rectangle(output, (0, 0), (label_width, 34), (0, 0, 0), -1)

    # Draw the label text in white on top of the black box
    cv2.putText(
        output,
        text,
        (10, 23),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (255, 255, 255),
        2,
    )

    # Return the labeled output image
    return output

```

## Assembling the Preview Panel

Nice work building up the toolkit with read_color, as_bgr, and label_image! Now it's time to tackle the height mismatch problem head-on by writing a helper that pads shorter images with black pixels so they can sit side by side with taller ones.

Open cvkit.py and complete the pad_to_height(image, height) function by following the TODOs:

    If the image's height already matches the target height, return it unchanged.
    Otherwise, build a black canvas with np.zeros using the shape (height, image.shape[1], 3) and the same dtype as image, and store it in output.
    Paste the original image into the top-left of output using array slicing on the height and width axes.
    Return the padded output.

This little helper is the key piece that allows np.hstack to line up panels of different sizes later on.

```
import cv2
import numpy as np


def read_color(path):
    image = cv2.imread(path, cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError(f"Could not read image: {path}")
    return image


def resize_long_edge(image, max_size=900):
    height, width = image.shape[:2]
    scale = min(1.0, max_size / max(height, width))
    if scale == 1.0:
        return image.copy()
    return cv2.resize(image, (int(width * scale), int(height * scale)))


def as_bgr(image):
    if image.ndim == 2:
        return cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    return image.copy()


def label_image(image, text):
    output = as_bgr(image)
    label_width = min(output.shape[1], max(220, 12 * len(text)))
    cv2.rectangle(output, (0, 0), (label_width, 34), (0, 0, 0), -1)
    cv2.putText(
        output,
        text,
        (10, 23),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (255, 255, 255),
        2,
    )
    return output


def pad_to_height(image, height):
    # TODO: If `image.shape[0]` already equals the target `height`,
    # return the image as is (no padding needed).

    # TODO: Otherwise, create a black canvas using np.zeros with shape
    # (height, image.shape[1], 3) and the same dtype as `image`.
    # Store it in a variable called `output`.

    # TODO: Paste the original `image` into the top-left of `output`
    # using array slicing on the height and width axes
    # (output[: image.shape[0], : image.shape[1]] = image).

    # TODO: Return the padded `output`.
    pass

```

Here is the completed `pad_to_height` function added to `cvkit.py`:

```python
import cv2
import numpy as np


def read_color(path):
    image = cv2.imread(path, cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError(f"Could not read image: {path}")
    return image


def resize_long_edge(image, max_size=900):
    height, width = image.shape[:2]
    scale = min(1.0, max_size / max(height, width))
    if scale == 1.0:
        return image.copy()
    return cv2.resize(image, (int(width * scale), int(height * scale)))


def as_bgr(image):
    if image.ndim == 2:
        return cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    return image.copy()


def label_image(image, text):
    output = as_bgr(image)
    label_width = min(output.shape[1], max(220, 12 * len(text)))
    cv2.rectangle(output, (0, 0), (label_width, 34), (0, 0, 0), -1)
    cv2.putText(
        output,
        text,
        (10, 23),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (255, 255, 255),
        2,
    )
    return output


def pad_to_height(image, height):
    # If image height already matches target height, return unchanged
    if image.shape[0] == height:
        return image

    # Otherwise, create a black canvas with target height and original width/channels
    output = np.zeros((height, image.shape[1], 3), dtype=image.dtype)

    # Paste the original image into the top-left of the canvas
    output[: image.shape[0], : image.shape[1]] = image

    # Return the padded canvas
    return output

```

## Showing Two Tiles Side by Side

With pad_to_height ready, every helper in the toolkit is finally in place. It is now time to bring them all together inside make_panel, the function that assembles a clean side-by-side preview from any list of named images.

Inside make_panel, follow the TODO comments to:

    Guard against empty input by raising a ValueError with the given message.
    Walk through each (name, image) pair, label it, resize it, and collect the result in tiles.
    Find the tallest tile and pad every other tile up to that height.
    Return the final panel built with np.hstack(tiles).

Once this is wired up, your cvkit.py toolkit will be complete and ready to power the stitching work coming next.

```
import cv2
import numpy as np


def read_color(path):
    image = cv2.imread(path, cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError(f"Could not read image: {path}")
    return image


def resize_long_edge(image, max_size=900):
    height, width = image.shape[:2]
    scale = min(1.0, max_size / max(height, width))
    if scale == 1.0:
        return image.copy()
    return cv2.resize(image, (int(width * scale), int(height * scale)))


def as_bgr(image):
    if image.ndim == 2:
        return cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    return image.copy()


def label_image(image, text):
    output = as_bgr(image)
    label_width = min(output.shape[1], max(220, 12 * len(text)))
    cv2.rectangle(output, (0, 0), (label_width, 34), (0, 0, 0), -1)
    cv2.putText(
        output,
        text,
        (10, 23),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (255, 255, 255),
        2,
    )
    return output


def pad_to_height(image, height):
    if image.shape[0] == height:
        return image

    output = np.zeros((height, image.shape[1], 3), dtype=image.dtype)
    output[: image.shape[0], : image.shape[1]] = image
    return output


def make_panel(items, max_size=900):
    # TODO: If `items` is empty (falsy), raise a ValueError with the message
    # "make_panel requires at least one image".

    # TODO: Initialize an empty list called `tiles`.

    # TODO: For each (name, image) pair in `items`, call label_image(image, name)
    # to stamp a label, then pass the result through resize_long_edge(..., max_size=max_size)
    # and append the resized labeled image to `tiles`.

    # TODO: Compute `max_height` as the maximum value of tile.shape[0] across all
    # tiles. Hint: use the max() function with a generator expression.

    # TODO: Replace `tiles` with a new list where every tile is padded to
    # `max_height` using pad_to_height.

    # TODO: Return np.hstack(tiles) to stitch the tiles side-by-side into one
    # large preview image.
    pass
```

Here is the completed `make_panel` function in `cvkit.py` that brings together labeling, resizing, height-padding, and horizontal stacking:

```python
import cv2
import numpy as np


def read_color(path):
    image = cv2.imread(path, cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError(f"Could not read image: {path}")
    return image


def resize_long_edge(image, max_size=900):
    height, width = image.shape[:2]
    scale = min(1.0, max_size / max(height, width))
    if scale == 1.0:
        return image.copy()
    return cv2.resize(image, (int(width * scale), int(height * scale)))


def as_bgr(image):
    if image.ndim == 2:
        return cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    return image.copy()


def label_image(image, text):
    output = as_bgr(image)
    label_width = min(output.shape[1], max(220, 12 * len(text)))
    cv2.rectangle(output, (0, 0), (label_width, 34), (0, 0, 0), -1)
    cv2.putText(
        output,
        text,
        (10, 23),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (255, 255, 255),
        2,
    )
    return output


def pad_to_height(image, height):
    if image.shape[0] == height:
        return image

    output = np.zeros((height, image.shape[1], 3), dtype=image.dtype)
    output[: image.shape[0], : image.shape[1]] = image
    return output


def make_panel(items, max_size=900):
    # Guard against empty input
    if not items:
        raise ValueError("make_panel requires at least one image")

    # Initialize empty list for tiles
    tiles = []

    # Label, resize, and collect each image tile
    for name, image in items:
        labeled = label_image(image, name)
        resized = resize_long_edge(labeled, max_size=max_size)
        tiles.append(resized)

    # Find the maximum height among all tiles
    max_height = max(tile.shape[0] for tile in tiles)

    # Pad every tile up to max_height
    tiles = [pad_to_height(tile, max_height) for tile in tiles]

    # Horizontally stack all tiles into a single preview panel
    return np.hstack(tiles)

```

Now that cvkit.py is fully assembled, it's time to take the brand-new make_panel for a spin and watch it work on a real picture.

The script already loads an image and sends one tile to make_panel, but a single tile isn't very interesting — the panel was designed to show things side by side.

Your job is to update the list passed to make_panel so it contains:

    Two (label, image) pairs
    The same loaded image in both pairs
    Two different label strings (for example, "original" and "copy")

When you run the script in a terminal with python solution.py sample_images/building/1.jpg, you should see the same picture displayed twice with two distinct labels stamped on top. This is a nice little proof that your toolkit really works!

```
import argparse
import cv2

from cvkit import make_panel, read_color


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("path")
    parser.add_argument("--preview-size", type=int, default=900)
    args = parser.parse_args()

    image = read_color(args.path)
    # TODO: Change the list below so it contains the same `image` twice,
    # each paired with a different label (for example "original" and "copy").
    # The panel will then display the image side-by-side with two labeled tiles.
    panel = make_panel([("original", image)], max_size=args.preview_size)

    cv2.imshow("workbench", panel)
    cv2.waitKey(0)
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()

```

Here is the updated `solution.py` script with the `make_panel` list updated to include two labeled tiles side-by-side:

```python
import argparse
import cv2

from cvkit import make_panel, read_color


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("path")
    parser.add_argument("--preview-size", type=int, default=900)
    args = parser.parse_args()

    image = read_color(args.path)

    # Pass two (label, image) pairs to display the image side-by-side
    panel = make_panel(
        [("original", image), ("copy", image)], 
        max_size=args.preview_size
    )

    cv2.imshow("workbench", panel)
    cv2.waitKey(0)
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()

```